# TrustLung AI — Model Training

This notebook walks through:
- Building MobileNetV2 and EfficientNetB0
- Two-phase training (feature extraction → fine-tuning)
- Plotting training curves
- Model comparison table
- Inference benchmarking

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import matplotlib.pyplot as plt
from src.utils.helpers import load_config, set_seed, get_device, print_metrics
from src.preprocessing.data_loader import generate_synthetic_dataset
from src.models.architectures import get_model, model_summary_dict
from src.models.trainer import LungCancerTrainer
from src.models.uncertainty import MCDropoutPredictor
from src.evaluation.metrics import compute_metrics, build_comparison_table
from src.visualization.plots import plot_training_curves
from src.benchmarking.inference_benchmark import InferenceBenchmarker

set_seed(42)
config = load_config('../configs/config.yaml')
device = get_device('auto')
print(f'Device: {device}')

data = generate_synthetic_dataset(n_samples=400, seed=42)
print(f'Data: train={len(data["X_train"])} val={len(data["X_val"])} test={len(data["X_test"])}')

## 1. Build Models

In [ ]:
mobilenet   = get_model('MobileNetV2',   input_shape=(224,224,3), num_classes=3, dropout_rate=0.4)
efficientnet = get_model('EfficientNetB0', input_shape=(224,224,3), num_classes=3, dropout_rate=0.4)

for name, m in [('MobileNetV2', mobilenet), ('EfficientNetB0', efficientnet)]:
    s = model_summary_dict(m)
    print(f'\n{name}:')
    for k, v in s.items():
        print(f'  {k}: {v}')

## 2. Train — EfficientNetB0 (Quick Demo)

In [ ]:
# For full training, use train.py
# Here we do 3 epochs as a demo
import tensorflow as tf

demo_model = get_model('EfficientNetB0', (224,224,3), 3, 0.4, pretrained=True)

history = demo_model.fit(
    data['X_train'][:100], data['y_train'][:100],
    validation_data=(data['X_val'][:30], data['y_val'][:30]),
    epochs=3, batch_size=16, verbose=1
)

# Wrap for plotting
hist_dict = {'phase1': history.history}
plot_training_curves(hist_dict, model_name='EfficientNetB0 (Demo)')
plt.show()

## 3. Evaluate with MC Dropout

In [ ]:
mc_pred   = MCDropoutPredictor(demo_model, n_samples=20, class_names=data['class_names'])
results   = mc_pred.predict_batch_with_uncertainty(data['X_test'][:50])
agg       = mc_pred.aggregate_batch_results(results)
metrics   = compute_metrics(data['y_test_raw'][:50], agg['y_pred'], agg['y_prob'], data['class_names'])
print_metrics(metrics, title='EfficientNetB0 (Demo) — Test Metrics')

## 4. Inference Benchmarking

In [ ]:
benchmarker = InferenceBenchmarker(n_warmup=3, n_runs=20)
result = benchmarker.benchmark_single_image(demo_model, data['X_test'][0], 'EfficientNetB0')

print('\nBenchmark Results:')
for k, v in result.items():
    print(f'  {k:<25} {v}')

# Batch size comparison
batch_df = benchmarker.benchmark_batch(
    demo_model, data['X_test'][:32],
    batch_sizes=[1, 4, 8, 16, 32],
    model_name='EfficientNetB0'
)
print('\nBatch Throughput:')
print(batch_df.to_string(index=False))